# Your first Python script, step by step

This notebook walks through exactly what `plot_stickleback.py` (the Lecture 03
warm-up script) does, one step at a time, so you can see what each line
contributes. The script and this notebook do the *same* work — the script runs
top-to-bottom with one command (`./plot_stickleback.py`), while here you run
each piece yourself and look at the intermediate results.

**To run in Positron:** open this `.ipynb` file, choose a Python interpreter if
prompted (top right), and run cells with `Shift+Enter`.

Don't worry about memorizing the pandas/matplotlib details — they get full
treatment in Lectures 05–07. Today the goal is to *see* that a script is just
these steps, saved in a file.

## Step 0 — Imports

A script says up front which packages it needs. `pandas` handles tables,
`matplotlib` draws plots. (`pathlib.Path` is the modern way to work with file
paths in Python — it knows how to check whether a file exists.)

Both packages are in the course environment; if an import fails, run
`pip install pandas matplotlib` in the terminal, then restart the kernel.

In [ ]:
from pathlib import Path

import pandas as pd                 # data tables    (L05-L06)
import matplotlib.pyplot as plt     # plotting       (L06)

## Step 1 — Find the data

The script shouldn't break just because you launched it from a different
folder. So it checks the usual course locations and, if the file isn't there,
falls back to downloading it from the course website — `pandas` can read a URL
exactly like a local path.

In [ ]:
candidates = [
    Path("data/Stickle_RNAseq.tsv"),            # from ~/projects/comptools
    Path("Data_Folder/Stickle_RNAseq.tsv"),     # from the repo root
    Path("../Data_Folder/Stickle_RNAseq.tsv"),  # from Examples_Folder
    Path("../../Data_Folder/Stickle_RNAseq.tsv"),  # from scripts_week03/
]
found = [p for p in candidates if p.exists()]
data_file = (
    found[0]
    if found
    else "https://wcresko.github.io/BioE_Comp_Tools_Fa2026/Data_Folder/Stickle_RNAseq.tsv"
)
data_file

## Step 2 — Read the tidy table

One line reads the whole thing. This works *only because the file is tidy*:

- a header line of variable names,
- one row per fish (the unit of observation),
- one value per cell,
- tab-separated (`sep="\t"`).

That's the Lecture 03 point in miniature: tools can make assumptions about
tidy data, so the code gets short.

In [ ]:
stickle = pd.read_csv(data_file, sep="\t")
print(f"{stickle.shape[0]} fish x {stickle.shape[1]} columns")
stickle.iloc[:5, :8]   # peek at the top-left corner

## Step 3 — One number per fish

We want each fish's **total** RNA-seq reads across the 600 gene columns.

- The list comprehension collects every column whose name starts with `"Gene"`.
- `.sum(axis=1)` sums *across* each row (axis=1 = per row; axis=0 would be per column).
- We keep the totals in a small two-column table next to each fish's group.
  (Yes, that awkward `Geno&Micro` column name — see the tidy-data discussion
  in Lecture 03 for why names like that will haunt you.)

In [ ]:
genes = [c for c in stickle.columns if c.startswith("Gene")]
print(f"{len(genes)} gene columns")

totals = pd.DataFrame({
    "Group": stickle["Geno&Micro"],
    "Total": stickle[genes].sum(axis=1),
})
totals.head()

## Step 4 — A quick plot

One box per genotype-×-microbiota group. pandas data frames have a built-in
`.boxplot()` that hands the drawing to matplotlib.

In [ ]:
totals.boxplot(column="Total", by="Group")
plt.ylabel("Total reads per fish")
plt.suptitle("")                     # drop pandas' default super-title
plt.title("Stickleback gut RNA-seq: total counts by group")
plt.show()

Look at the plot before moving on: which group stands out? (We'll ask
*why* — and whether it's biology or sequencing depth — when we get to
normalization later in the course.)

## Step 5 — Save the figure to a file

In a notebook you can see the plot above, but a *script* has no screen — it
saves the figure to a file instead. This cell does what the script does:
makes a `results/` folder if needed and writes a PNG there.

In [ ]:
out_dir = Path("results")
out_dir.mkdir(exist_ok=True)
out_file = out_dir / "total_counts_by_group.png"

totals.boxplot(column="Total", by="Group")
plt.ylabel("Total reads per fish")
plt.suptitle("")
plt.title("Stickleback gut RNA-seq: total counts by group")
plt.savefig(out_file, dpi=200)
plt.close()                          # done drawing; close the figure
print(f"Wrote {out_file}")

## That's the whole script

Every cell above corresponds to a numbered section in
`plot_stickleback.py`. The script adds exactly two things this notebook
doesn't need:

1. a **shebang** on line 1 (`#!/usr/bin/env python3`) so the shell knows
   which program runs the file, and
2. `matplotlib.use("Agg")` so it can draw with no window open.

Then `chmod +x` once, and `./plot_stickleback.py` re-runs the entire analysis
with one command — that's the reproducibility habit from Lectures 02–03.

**Keep going:** Python scripting starts in earnest in
[Lecture 05](https://wcresko.github.io/BioE_Comp_Tools_Fa2026/Lecture_Folder/Lecture_05_BioE_Comp_Tools_2026.html),
and the [Example Files](https://wcresko.github.io/BioE_Comp_Tools_Fa2026/Examples.html)
page has a set of Python tutorial notebooks to practice with.